In [0]:
from pyspark.sql.functions import (col,trim,upper,to_date,round,current_date,datediff,date_trunc,expr,when, date_add)

In [0]:
from src.claims_project.config import TABLES

In [0]:
bronze_claims = spark.read.table(TABLES['bronze_claims'])
display(bronze_claims.limit(5))

In [0]:
silver_claims = bronze_claims\
    .withColumn("ClaimID", trim(col("ClaimID")))\
    .dropDuplicates(["ClaimID"])\
    .withColumn("PatientID", trim(col("PatientID")))\
    .withColumn("ProviderID", trim(col("ProviderID")))\
    .withColumn("DiagnosisCode", trim(upper(col("DiagnosisCode"))))\
    .withColumn("ProcedureCode", trim(upper(col("ProcedureCode"))))\
    .withColumn("ProviderSpecialty", upper(col("ProviderSpecialty")))\
    .withColumn("ProviderLocation", upper(col("ProviderLocation")))\
    .withColumn("ClaimStatus", upper(col("ClaimStatus")))\
    .withColumn("ClaimType", trim(col("ClaimType")))\
    .withColumn("ClaimDate", to_date(col("ClaimDate"), "dd-MM-yyyy"))\
    .withColumn("ClaimAmount", round(col("ClaimAmount").cast("double"),2))\
    .withColumn("claim_received_date", date_add(col("ClaimDate"), 5))\
    .withColumn("paid_date", date_add(col("ClaimDate"), 15))\
    .withColumn("claim_month", date_trunc("MONTH", col("ClaimDate")))\
    .withColumn("claim_lag_days", datediff(col("claim_received_date"), col("ClaimDate")))\
    .withColumn("payment_lag_days", datediff(col("paid_date"), col("ClaimDate")))\
    .filter(col("ClaimID").isNotNull())\
    .filter(col("PatientID").isNotNull())\
    .filter(col("ProviderID").isNotNull())\
    .filter(col("ClaimAmount").isNotNull())

display(silver_claims.limit(5))

In [0]:
silver_claims.write.format("delta").mode("overwrite").option("mergeschema", "true").saveAsTable(TABLES['silver_claims'])